In [1]:
import os
os.environ["OPENBLAS_NUM_THREADS"] = "1"


In [ ]:
# 1. Gerekli kütüphaneleri yükle
from pyspark.sql import SparkSession
from pyspark.ml.clustering import KMeans
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.evaluation import ClusteringEvaluator
import pandas as pd
from sklearn.metrics import silhouette_samples
import numpy as np
import matplotlib.pyplot as plt

# 2. Spark Session başlat
spark = SparkSession.builder \
    .appName("KMeans_Silhouette_Per_Cluster") \
    .getOrCreate()

# 3. Veriyi oku
df = spark.read.csv('5G_DL_DL_Filled.csv', header=True, inferSchema=True)

# 4. Longitude ve Latitude seç
data = df.select('Longitude', 'Latitude')

# 5. Özellikleri birleştir (VectorAssembler)
assembler = VectorAssembler(inputCols=["Longitude", "Latitude"], outputCol="features")
feature_data = assembler.transform(data)

# 6. KMeans model oluştur ve eğit
kmeans = KMeans(k=500, seed=42)  # Örneğin 500 cluster
model = kmeans.fit(feature_data)

# 7. Küme tahminleri yap
predictions = model.transform(feature_data)

# 8. Performansı değerlendir (genel silhouette skoru)
evaluator = ClusteringEvaluator()
silhouette_score_total = evaluator.evaluate(predictions)
print(f"Genel Silhouette Score: {silhouette_score_total:.4f}")

# 9. Spark DataFrame'i Pandas DataFrame'e çevir
predictions_pd = predictions.select('Longitude', 'Latitude', 'prediction').toPandas()

# 10. Özellikler ve Etiketler
X = predictions_pd[['Longitude', 'Latitude']].values
labels = predictions_pd['prediction'].values

# 11. Her örnek için silhouette skoru hesapla
sample_silhouette_values = silhouette_samples(X, labels)

# 12. Küme başına ortalama silhouette ve satır sayısı hesapla
cluster_ids = np.unique(labels)
avg_silhouette_per_cluster = []
count_per_cluster = []

print("\nKüme Başına Ortalama Silhouette Skorları ve Satır Sayıları:")
for cluster_label in cluster_ids:
    cluster_silhouette_values = sample_silhouette_values[labels == cluster_label]
    avg_cluster_silhouette = cluster_silhouette_values.mean()
    avg_silhouette_per_cluster.append(avg_cluster_silhouette)
    
    cluster_size = (labels == cluster_label).sum()
    count_per_cluster.append(cluster_size)
    
    print(f"Küme {cluster_label}: Ortalama Silhouette = {avg_cluster_silhouette:.4f}, Satır Sayısı = {cluster_size}")

# 13. Küme Başına Satır Sayısı Grafiği
plt.figure(figsize=(12, 6))
plt.bar(cluster_ids, count_per_cluster)
plt.title('Her Kümedeki Satır Sayısı')
plt.xlabel('Küme ID')
plt.ylabel('Satır Sayısı')
plt.grid(True)
plt.show()

# 14. (İstersen ayrıca Silhouette skorlarını da çizebiliriz)
plt.figure(figsize=(12, 6))
plt.bar(cluster_ids, avg_silhouette_per_cluster, color='green')
plt.title('Her Kümedeki Ortalama Silhouette Skoru')
plt.xlabel('Küme ID')
plt.ylabel('Ortalama Silhouette Skoru')
plt.grid(True)
plt.show()

# 15. SparkSession'ı kapat
spark.stop()


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/04/12 14:12:20 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/04/12 14:12:43 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS


Genel Silhouette Score: 0.7257
